In [36]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [37]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

In [38]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

In [39]:
def call_openai(model, messages, temperature=0.7):
    client = OpenAI(
        base_url=MODEL_MAP[model]["endpoint"],
        api_key=MODEL_MAP[model]["api_key"]
    )
    response = client.chat.completions.create(
        model=MODEL_MAP[model]["model"],
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


In [40]:
SYSTEM_PROMPT = """
You are an expert medical data scientist generating high-fidelity, synthetic patient health records for research purposes.
Your goal is to create data that mimics realistic, complex health patterns, including correlations between lifestyle, demographics, and clinical lab values.
The output must be in JSON or comma-separated CSV format.
Ensure that all data points are plausible (e.g., age 80 does not have a resting heart rate of 30, BMI of 40 correlates with high blood pressure, etc.).
Do not include any real patient identifiers (PHI).
Generate 20 distinct, uncorrelated features per record.
"""

USER_PROMPT = """
Role: Act as an expert data scientist and clinical informaticist specializing in generating high-fidelity, privacy-compliant synthetic healthcare data.
Task: Generate a CSV-formatted dataset of 100 synthetic patient records. 
Dataset Specifications:

    Target: Individuals' health conditions and associated metrics.
    Format: CSV, comma-separated.
    Privacy: No real PII (Personally Identifiable Information). Use realistic but artificial data.
    Data Types: Include a mix of numerical, categorical, and binary variables.
    Realism: Ensure logical correlations between features (e.g., higher BMI correlates with higher blood pressure, older age correlates with higher prevalence of chronic conditions). 

Features to Include (20 total):

    Patient_ID (Unique alphanumeric)
    Age (Integer, 18-90)
    Gender (Categorical: Male, Female, Other)
    Height_cm (Numerical)
    Weight_kg (Numerical)
    BMI (Calculated)
    BloodPressure_Systolic (Numerical)
    BloodPressure_Diastolic (Numerical)
    HeartRate_BPM (Numerical)
    Smoking_Status (Categorical: Never, Former, Current)
    Alcohol_Consumption (Categorical: Low, Medium, High)
    Physical_Activity_Hours_Per_Week (Numerical)
    HbA1c_Level (Numerical, 4.0 - 10.0)
    LDL_Cholesterol_mgdL (Numerical)
    Daily_Sleep_Hours (Numerical)
    Known_Allergies (Categorical: None, Penicillin, Peanuts, Pollen)
    Primary_Condition (Categorical: Hypertension, Type 2 Diabetes, Asthma, None)
    Number_of_Medications (Integer)
    Mental_Health_Score_PHQ9 (Integer, 0-27)
    Hospital_Readmission_30d (Binary: 0 or 1) 

Output Format:
Provide the output directly in CSV format.
"""

messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ]

In [41]:
def generate_dataset(model):
    return call_openai(model, messages)

In [52]:
csv_string = generate_dataset("GPT")
csv_string = csv_string.replace("```", "").replace("csv", "").strip()
print(csv_string)

Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BloodPressure_Systolic,BloodPressure_Diastolic,HeartRate_BPM,Smoking_Status,Alcohol_Consumption,Physical_Activity_Hours_Per_Week,HbA1c_Level,LDL_Cholesterol_mgdL,Daily_Sleep_Hours,Known_Allergies,Primary_Condition,Number_of_Medications,Mental_Health_Score_PHQ9,Hospital_Readmission_30d
P001,45,Male,175,85,27.76,130,85,72,Former,Medium,3,5.8,130,6.5,None,Hypertension,3,10,0
P002,32,Female,165,55,20.20,120,75,68,Never,Low,5,5.4,100,7.0,Pollen,None,1,5,0
P003,60,Other,180,90,27.78,150,90,80,Former,High,2,6.5,160,5.0,Peanuts,Type 2 Diabetes,2,12,1
P004,29,Male,170,65,22.49,118,76,75,Never,Low,4,5.2,110,8.0,None,Asthma,1,4,0
P005,54,Female,160,70,27.48,140,88,78,Current,Medium,1,7.0,150,6.0,Pollen,Hypertension,3,15,0
P006,75,Male,175,95,31.02,160,95,70,Former,High,1,8.2,180,5.5,Peanuts,Type 2 Diabetes,4,18,1
P007,40,Female,168,60,21.26,122,78,65,Never,Low,6,5.5,90,7.5,None,None,2,3,0
P008,50,Male,182,110,33.24,145,92,72,Current,High,0,6.8,175,5.0

In [ ]:
import pandas as pd
import io

# Sample data
csv_data_stream = io.StringIO(csv_string)
# df = pd.DataFrame(csv_data)

csv_data = None

def load_csv_data():
    global csv_data

    csv_data = pd.read_csv(csv_data_stream)

load_csv_data()

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import datetime
import warnings
import tempfile
from cachetools import cached, TTLCache

warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")


cache = TTLCache(maxsize=128, ttl=300)

@cached(cache)
def get_gender_categories():
    global csv_data
    if csv_data is None:
        return []
    cats = sorted(csv_data['Gender'].dropna().unique().tolist())
    cats = [cat.capitalize() for cat in cats]
    return cats



['Female', 'Male', 'Other']